# API Examples — the `car_pricing` package in action

Runnable companion to [`docs/API_EXAMPLES.md`](../docs/API_EXAMPLES.md).

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
for p in ('.', 'src'):
    sys.path.insert(0, os.path.abspath(p))

## Predict

In [2]:
from car_pricing.predict import predict
predict({'make':'MARUTI','model':'SWIFT VXI','age':5,'km_driven':40000})

{'predicted_price_lakhs': 5.74,
 'predicted_price_display': '₹5.74 Lakhs',
 'price_band': 'Medium'}

## Load + validate the data

In [3]:
from car_pricing import data, validation
df = data.clean(data.load_raw())
print(df.shape)
print(validation.validate_dataframe(df).summary())

(19820, 17)
Data validation on 19,820 rows: PASS
  [warning] 'km_driven' has 3 value(s) outside [0, 1000000]
  [warning] 'mileage' has 4 value(s) outside [0.0, 60.0]


## Features — bands

In [4]:
from car_pricing import features
X, y = features.split_xy(df)
edges = features.band_edges(y)
print('edges:', edges)
print(features.price_to_band([2.0, 5.0, 25.0], edges))

edges: [0.3, 3.99, 6.75, 20.9]
['Low' 'Medium' 'High']


## Explainability — top features

In [5]:
from car_pricing import explain
explain.builtin_importances().head(6).round(3)

model        7365
km_driven    7308
make         5046
mileage      4715
max_power    4569
age          3636
dtype: int32

## Drift detection

In [6]:
from car_pricing import drift
ref = df.sample(frac=0.5, random_state=1)
cur = df.sample(frac=0.5, random_state=2).copy(); cur['age'] += 8
out = drift.detect_drift(ref, cur)
print('drift:', out['drift_detected'], '| n:', out['n_drifted'])
out['report']

drift: True | n: 1


,feature,psi,ks_pvalue,drift
0,km_driven,0.001,0.9327,False
1,mileage,0.001,0.9947,False
2,engine,0.001,0.5284,False
3,max_power,0.002,0.7411,False
4,age,10.055,0.0000,True


## Monitoring

In [7]:
from car_pricing import monitoring
monitoring.log_prediction({'make':'MARUTI','model':'SWIFT VXI'}, 5.74, actual_lakhs=5.5)
monitoring.live_metrics()

{'n_predictions': 4,
 'n_with_actual': 4,
 'live_mae_lakhs': 0.588,
 'kpi_breached': False}

## Serving — the Flask test client (no server needed)

In [8]:
from car_pricing.serve import app
c = app.test_client()
print(c.get('/health').get_json())
print(c.post('/predict', json={'make':'MARUTI','model':'SWIFT VXI'}).get_json())

{'status': 'healthy'}
{'predicted_price_display': '₹4.81 Lakhs', 'predicted_price_lakhs': 4.81, 'price_band': 'Medium'}


See `docs/API_EXAMPLES.md` for tuning, tracking, registry and full training.